# Bayesian Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChemAI-Lab/AI4Chem/blob/main/website/modules/06-bayesian_optimization.ipynb)

**References:**
1. **Chapters 1-3**: [Bayesian Optimization](https://bayesoptbook.com/book/bayesoptbook.pdf), R. Garnett
2. **Chapters 6**: [Pattern Recognition and Machine Learning](https://www.microsoft.com/en-us/research/wp-content/uploads/2006/01/Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf), C. M. Bishop.
3. **Chapter 2**:  [Gaussian Processes for Machine Learning](https://direct.mit.edu/books/oa-monograph-pdf/2514321/book_9780262256834.pdf), C. E. Rasmussen, C. K. I. Williams
4. **Chapter 4**: [Machine Learning in Quantum Sciences](https://arxiv.org/pdf/2204.04198)
5. **Chapter 6**: [Probabilistic Machine Learning: An Introduction, K. P. Murphy.](https://probml.github.io/pml-book/book1.html)
6. [**The Kernel Cookbook**](https://www.cs.toronto.edu/~duvenaud/cookbook/)

**Power Point Slides**: [![Bayesian Optimization Slides](https://img.shields.io/badge/Slides–Download-PPTX-success?logo=microsoftpowerpoint&logoColor=white)](https://raw.githubusercontent.com/ChemAI-Lab/AI4Chem/main/website/modules/BayesOpt.pptx)



# Optimization without Gradients

During the course, we saw that many scientific problems can be recast as an optimization problem, 
$$
\mathbf{x}^* = \arg\min f(\mathbf{x})
$$
where $\mathbf{x}^*$ is the **minimizer** of the function $f$. <br>
We have solved this problem using gradient-based methods, where at each step we used the local information of the gradient to move "towards" the minimizer of $f$, using
$$
\mathbf{x}^*_{t+1} = \mathbf{x}^*_{t} - \eta \nabla_{\mathbf{x}}f.
$$
These style of methods have been successful in scientific frameworks where $f$ is differentiable, or its gradient can be easily estimated. However, for other systems where $f$ is a **black box** function, gradient-based optimization is unfeasible. 

## Black Box Function
* A black box function is a system, algorithm, or piece of code where only the inputs and outputs are visible, while the internal logic, mechanisms, or code structure are hidden or unknown.

```
x ∈ ℝ^d   ─────▶   [   BLACK BOX  f(x)   ]   ─────▶   y = f(x)
(parameters)                                   (objective value)
```

*  Expensive (minutes–days) 
*  No gradients available  
*  Noisy observations  <br>


> In **Bayesian Optimization**, we assume we can query the function, but we cannot inspect its internals. <br>
> We cannot differentiate it analytically, and each evaluation may cost minutes, hours, or even days. <br>
> Therefore, we must be strategic about where to sample next.


Any Bayesian Optimization algorithm is composed of three main ingredients,
1. **Surrogate models** --> Approximates $f(\mathbf{x}$)
2. **Acquisition functions** --> Quantifies the information gain if a new point is known
3. **Exploration vs exploitation** --> Uncertainty vs Certainty

In [ ]:
# !pip install py3Dmol
# !pip install rdkit
# !pip install pyscf
# !pip install botorch

In [ ]:
from botorch.acquisition import ExpectedImprovement
from botorch.acquisition import UpperConfidenceBound
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.fit import fit_gpytorch_mll
from botorch.models import SingleTaskGP
import torch
import pyscf
import numpy as np
import py3Dmol
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw, rdDetermineBonds, MolFromXYZBlock
from rdkit.Chem import rdDetermineBonds
from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_3d = True

from pyscf import dft

from botorch.optim import optimize_acqf

import matplotlib.pyplot as plt

In [ ]:
def objective(x):
	return np.sin(x) + np.sin((10.0 / 3.0) * x)

# define optimal input value
x_optima = 5.145735
# hola

## Surrogate model

Since the evaluations of $f$ are expensive, we want to make as few function evaluations as possible. 
This suggest the need to approximate $f$ with a model, which in the Bayesian optimization literature is known as **surrogate model**, $f_{\mathbf{\theta}}$, and will be trained with the collected data.


In [ ]:
x = np.random.uniform(0, 10, size=(5, 1))
y = objective(x)    

x_grid = np.linspace(0, 10, 100).reshape(-1, 1)
y_grid = objective(x_grid)  

In [ ]:
from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize

train_x = torch.tensor(x).float()
train_y = torch.tensor(y).float()
print(train_x.shape, train_y.shape)

gp_model = SingleTaskGP(
    train_x,
    train_y,
    input_transform=Normalize(d=train_x.shape[-1]),
)
mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model)
fit_gpytorch_mll(mll)

In [ ]:
plt.plot(x_grid, y_grid, label='Objective Function')
plt.scatter(x, y, color='k', marker='x', label='Initial Samples')
plt.legend()